# ETL VRA/ANAC 2019 — Contrato v1.1

Este notebook executa o **ETL de 2019** com as regras do Contrato Único de Dados e Tratamento v1.1 (11/11/2026). A lógica central fica no arquivo `etl_vra_anac_2019.py`; assim, a execução pelo notebook e pelo terminal utiliza as mesmas regras.

## Regras importantes desta versão

- Preserva os 12 CSVs originais e lê os 20 campos como texto.
- Deduplica **antes da padronização**, comparando exatamente os 20 valores brutos. Mantém a primeira ocorrência por arquivo/linha e audita as excedentes.
- Sinaliza possíveis duplicidades que só aparecem após `TRIM`/`UPPER`, sem removê-las.
- Rejeita somente quando falta um dos cinco campos obrigatórios: companhia ICAO, aeroporto de origem, aeroporto de destino, partida prevista ou chegada prevista.
- Campos opcionais inválidos geram anomalias, sem rejeição por si só.
- Gera as sete dimensões, a Fato, auditorias e resumo de qualidade.

**Execução recomendada:** VS Code/Jupyter no computador que tem acesso ao PostgreSQL. O banco `vra_dw_2019` precisa existir antes da carga. No Google Colab, `localhost` aponta para a máquina virtual do Colab, não para o PostgreSQL do computador; nesse caso, seria necessário configurar um banco remoto acessível.


In [ ]:
# Instala as dependências do projeto no ambiente atual do Jupyter.
%pip install -q "pandas>=2.2,<3.0" "numpy>=1.26,<3.0" "SQLAlchemy>=2.0,<3.0" "psycopg[binary]>=3.2,<4.0" "python-dotenv>=1.0,<2.0"

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from dotenv import load_dotenv

# Procura a raiz do projeto na pasta atual ou em alguma pasta-pai.
candidatos = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next(
    (p for p in candidatos if (p / "etl_vra_anac_2019.py").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_DIR))
SCRIPT_PATH = PROJECT_DIR / "etl_vra_anac_2019.py"
assert SCRIPT_PATH.exists(), (
    f"Não encontrei {SCRIPT_PATH}. Coloque o notebook e o script na raiz do projeto "
    "ou ajuste PROJECT_DIR manualmente."
)

load_dotenv(PROJECT_DIR / ".env", override=False)
INPUT_DIR = PROJECT_DIR / "dados" / "brutos" / "2019"
OUTPUT_DIR = PROJECT_DIR / "dados" / "processados" / "2019"

print("Pasta do projeto:", PROJECT_DIR)
print("Script:", SCRIPT_PATH)
print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)

## 1. Conferência dos arquivos e cabeçalhos

O contrato exige os 12 arquivos mensais e 20 colunas originais. Esta célula confere os nomes, a ordem e a quantidade, sem carregar o conjunto inteiro na memória.


In [ ]:
from etl_vra_anac_2019 import ORIGINAL_COLS

arquivos_esperados = [f"VRA_2019_{mes:02d}.csv" for mes in range(1, 13)]
arquivos_encontrados = sorted(p.name for p in INPUT_DIR.glob("VRA_2019_*.csv"))
assert arquivos_encontrados == arquivos_esperados, (
    f"Arquivos divergentes. Ausentes: {sorted(set(arquivos_esperados)-set(arquivos_encontrados))}; "
    f"extras: {sorted(set(arquivos_encontrados)-set(arquivos_esperados))}"
)

conferencia = []
for nome in arquivos_esperados:
    caminho = INPUT_DIR / nome
    cabecalho = pd.read_csv(
        caminho, sep=";", encoding="utf-8-sig", dtype=str,
        keep_default_na=False, nrows=0,
    ).columns.tolist()
    assert cabecalho == ORIGINAL_COLS, f"Cabeçalho divergente em {nome}"
    conferencia.append({"arquivo": nome, "tamanho_bytes": caminho.stat().st_size, "colunas": len(cabecalho)})

display(pd.DataFrame(conferencia))
print("Conferência concluída: 12 arquivos e 20 colunas em cada arquivo.")

## 2. Configuração do PostgreSQL

Crie previamente o banco `vra_dw_2019` no pgAdmin. Configure as credenciais em um arquivo `.env` na raiz do projeto, usando `.env.example` como modelo. Não coloque senha diretamente neste notebook ou no GitHub.


In [ ]:
load_dotenv(PROJECT_DIR / ".env", override=False)

variaveis_pg = ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]
if not os.getenv("DATABASE_URL"):
    ausentes = [nome for nome in variaveis_pg if not os.getenv(nome)]
    print("Variáveis ausentes:", ausentes if ausentes else "nenhuma")
else:
    print("DATABASE_URL configurada (valor oculto).")

print("Host:", os.getenv("PGHOST", "definido em DATABASE_URL"))
print("Porta:", os.getenv("PGPORT", "definida em DATABASE_URL"))
print("Banco:", os.getenv("PGDATABASE", "definido em DATABASE_URL"))
print("Usuário:", os.getenv("PGUSER", "definido em DATABASE_URL"))
print("Schema:", os.getenv("PGSCHEMA", "dw_etl"))
print("Senha configurada:", "sim" if (os.getenv("PGPASSWORD") or os.getenv("DATABASE_URL")) else "não")

## 3. Execução do ETL

Por segurança, `REPLACE_CARGA_EXISTENTE` começa como `False`. Se as tabelas já existirem no schema, a execução será interrompida para evitar duplicação. Altere para `True` somente quando desejar apagar e recriar as tabelas desta carga.

Para testar a transformação sem PostgreSQL, defina `EXECUTAR_CARGA_POSTGRES = False`. A execução padrão abaixo está preparada para carregar o PostgreSQL.


In [ ]:
from etl_vra_anac_2019 import run_etl

EXECUTAR_CARGA_POSTGRES = True
REPLACE_CARGA_EXISTENTE = False

if EXECUTAR_CARGA_POSTGRES and not (os.getenv("DATABASE_URL") or os.getenv("PGPASSWORD")):
    raise RuntimeError("Configure a conexão no .env antes de executar a carga PostgreSQL.")

resultado = run_etl(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    load_database=EXECUTAR_CARGA_POSTGRES,
    replace=REPLACE_CARGA_EXISTENTE,
    schema=os.getenv("PGSCHEMA", "dw_etl"),
)
print(json.dumps(resultado, ensure_ascii=False, indent=2, default=str))

## 4. Conferência da conciliação e indicadores

A equação obrigatória é: `linhas brutas = ocorrências excedentes excluídas + rejeições remanescentes + linhas elegíveis na Fato`. A interseção entre excedentes duplicados e registros inválidos é apresentada em indicador separado e não é subtraída duas vezes.


In [ ]:
resumo_path = OUTPUT_DIR / "etl_resumo.json"
assert resumo_path.exists(), "O arquivo etl_resumo.json não foi gerado."
resumo = json.loads(resumo_path.read_text(encoding="utf-8"))

conciliacao = resumo["equacao_conciliacao"]
assert conciliacao["conciliada"], "A equação de conciliação não fechou."
assert conciliacao["brutas"] == (
    conciliacao["excedentes_excluidas"]
    + conciliacao["rejeicoes_remanescentes"]
    + conciliacao["fato"]
)

indicadores_principais = [
    "linhas_brutas",
    "grupos_duplicados",
    "linhas_envolvidas_duplicatas_exatas",
    "ocorrencias_excedentes_excluidas",
    "grupos_pos_trim_upper",
    "registros_envolvidos_pos_trim_upper",
    "intersecao_excedentes_duplicados_invalidos",
    "linhas_rejeitadas_pos_deduplicacao",
    "linhas_elegiveis_fato",
    "voos_cancelados_na_fonte",
    "voos_cancelados_na_fato",
    "partidas_atipicas_abs_maior_24h",
    "chegadas_atipicas_abs_maior_24h",
]
display(pd.DataFrame([{"indicador": k, "valor": resumo.get(k)} for k in indicadores_principais]))
print("Conciliação aprovada.")

## 5. Conferência das tabelas no PostgreSQL

A carga cria as tabelas de preservação, auditoria, qualidade, log, dimensões e Fato no schema configurado. O script também verifica as chaves estrangeiras durante a transação.


In [ ]:
if resumo.get("carga_postgresql") == "concluida":
    from sqlalchemy import text
    from etl_vra_anac_2019 import _postgres_engine, TABLES

    schema = os.getenv("PGSCHEMA", "dw_etl")
    assert schema.replace("_", "").isalnum(), "Nome de schema inválido para consulta."
    engine = _postgres_engine()
    try:
        with engine.connect() as con:
            contagens = pd.DataFrame([
                {
                    "tabela": tabela,
                    "registros": con.execute(
                        text(f'SELECT COUNT(*) FROM "{schema}"."{tabela}"')
                    ).scalar_one(),
                }
                for tabela in TABLES
            ])
        display(contagens)
        assert resumo.get("foreign_key_orphans") == 0
    finally:
        engine.dispose()
else:
    print("Carga PostgreSQL não executada nesta sessão. Os CSVs e auditorias foram gerados localmente.")

## 6. Arquivos produzidos

A pasta `dados/processados/2019/` contém a cópia bruta, a camada tratada, a Fato, as sete dimensões, as auditorias, os logs e o resumo JSON. A próxima célula lista os arquivos efetivamente gerados.


In [ ]:
assert OUTPUT_DIR.exists(), f"Pasta de saída não encontrada: {OUTPUT_DIR}"
for caminho in sorted(OUTPUT_DIR.rglob("*")):
    if caminho.is_file():
        print(f"{caminho.relative_to(OUTPUT_DIR)} — {caminho.stat().st_size:,} bytes")

## 7. Referências e limites do escopo

O contrato v1.1 apresenta números de exploração de 2019 (por exemplo, 983.414 linhas brutas, 444 grupos repetidos e 470 excedentes), mas determina que a nova versão seja executada para recalcular as contagens e medir a interseção com rejeições. Esses números são referências de conferência, não valores fixados no código.

Este notebook executa somente o ETL de 2019. O ELT, os anos de 2020/2021, a decisão sobre fuso horário, a semântica de `referencia` e o limiar comum para classificar voos atrasados permanecem no escopo coletivo do grupo.
